# COSC83 — Super-Resolution Training (Colab)

Scale factor **2×** · DIV2K dataset · ResNet-style SRCNN

**Resume-safe**: checkpoints are written to Google Drive. If the runtime
disconnects mid-training, re-run the *Train* cell — it loads the last best
checkpoint automatically and continues from that epoch.

| # | Cell | Notes |
|---|------|-------|
| 1 | Mount Google Drive | Run once per session |
| 2 | Clone / pull repo | Needs your GitHub URL (see TODO) |
| 3 | Install dependencies | Fast on Colab (torch pre-installed) |
| 4 | Download DIV2K | ~4 GB total; skips if already present |
| 5 | **Train** | 50 epochs, scale×2; auto-resumes from Drive |
| 6 | **Evaluate** | SR vs. bicubic — prints final PSNR / SSIM |
| 7 | Plot training curves | Reads `training_history.png` from Drive |

> **Before running**: *Runtime → Change runtime type → T4 GPU*

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
# Checkpoints and sample images are written here so they survive
# runtime disconnects. Re-mount at the start of every new session.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/COSC83_SR_scale2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir ready: {DRIVE_DIR}')

In [ ]:
# ── Cell 2: Clone / pull repo ────────────────────────────────────────────────
# TODO: replace with your actual GitHub repo URL before running
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/COSC83-spring24-25-student.git'
REPO_DIR = '/content/COSC83-spring24-25-student'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo already present — pulled latest.')

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# torch, torchvision, numpy, pillow, matplotlib are pre-installed on Colab.
# Only tqdm and headless OpenCV need adding.
!pip install -q tqdm opencv-python-headless
print('Dependencies ready.')

In [ ]:
# ── Cell 4: Download DIV2K train + val HR images (~4 GB total) ───────────────
# Uses download_data.py from the repo root.
# Skips automatically if the directories already exist.
# Expect ~5–10 min on a fresh runtime.
import os
REPO_DIR = '/content/COSC83-spring24-25-student'
os.chdir(REPO_DIR)
!python download_data.py

In [ ]:
# ── Cell 5: Train — 50 epochs, scale×2 ──────────────────────────────────────
# Checkpoints go to Google Drive; auto-resumes if best_model.pth is present.
# Expected wall time on T4: ~35–45 min for 50 epochs at batch=16, patch=128.
import os, sys

REPO_DIR       = '/content/COSC83-spring24-25-student'
DRIVE_DIR      = '/content/drive/MyDrive/COSC83_SR_scale2'  # redefine in case Cell 1 was skipped
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
SAMPLE_DIR     = os.path.join(DRIVE_DIR, 'samples')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR,     exist_ok=True)

# Work from assignment1/ so relative paths like '../data/DIV2K/' resolve correctly
os.chdir(os.path.join(REPO_DIR, 'assignment1'))
sys.path.insert(0, os.path.join(REPO_DIR, 'assignment1'))

from train import train

best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

config = {
    # Model
    'scale_factor':  2,
    'num_features':  64,
    'num_blocks':    16,

    # Data — relative to assignment1/ working directory
    'train_dir':  '../data/DIV2K/DIV2K_train_HR',
    'val_dir':    '../data/DIV2K/DIV2K_valid_HR',
    'patch_size': 128,
    'downsample_methods': ['bicubic', 'bilinear', 'nearest', 'lanczos'],

    # Training
    'batch_size':          16,
    'num_epochs':          50,
    'learning_rate':       1e-4,
    'lr_decay_step':       30,    # halve LR at epoch 30
    'lr_decay_gamma':      0.5,
    'num_workers':         2,     # Colab restricts multiprocessing; keep ≤ 2
    'validation_interval': 5,     # validate every 5 epochs
    'val_batch_limit':     20,    # cap val batches for speed

    # Bonus option B: set True to add VGG perceptual loss on top of L1
    'use_perceptual_loss':    False,
    'perceptual_loss_weight': 0.1,

    # Checkpoints go to Drive
    'checkpoint_dir': CHECKPOINT_DIR,
    'sample_dir':     SAMPLE_DIR,
    'save_every':     5,

    # Auto-resume: if best_model.pth exists on Drive, continue from that epoch.
    # On a fresh run this is None; after a disconnect it picks up automatically.
    'resume': best_ckpt if os.path.exists(best_ckpt) else None,
}

if config['resume']:
    import torch as _t
    _ck    = _t.load(best_ckpt, map_location='cpu')
    _epoch = _ck.get('epoch', '?')
    _psnr  = _ck.get('psnr',  0.0)
    print(f'Resuming from epoch {_epoch}  (best PSNR so far: {_psnr:.2f} dB)')
else:
    print('Starting fresh training run.')

train(config)

In [ ]:
# ── Cell 6: Evaluate — SR model vs. bicubic baseline ────────────────────────
# Loads the best checkpoint from Drive and evaluates on the full val set
# using 256×256 patches (more representative than the 128×128 training patches).
import os, sys
import torch
import torch.nn.functional as F

REPO_DIR       = '/content/COSC83-spring24-25-student'
DRIVE_DIR      = '/content/drive/MyDrive/COSC83_SR_scale2'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

os.chdir(os.path.join(REPO_DIR, 'assignment1'))
if os.path.join(REPO_DIR, 'assignment1') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'assignment1'))

from srcnn      import SuperResolutionCNN
from dataloader import get_dataloader
from metrics    import fast_psnr, fast_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load best checkpoint ─────────────────────────────────────────────────────
best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
if not os.path.exists(best_ckpt):
    raise FileNotFoundError(
        f'No checkpoint at {best_ckpt} — run the Train cell first.')

ckpt  = torch.load(best_ckpt, map_location=device)
model = SuperResolutionCNN(
    scale_factor=2, num_channels=3, num_features=64, num_blocks=16
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

_epoch = ckpt.get('epoch', '?')
_psnr  = ckpt.get('psnr',  0.0)
print(f'Loaded checkpoint: epoch {_epoch},  training best PSNR {_psnr:.2f} dB')

# ── Validation loader ────────────────────────────────────────────────────────
# 256×256 patches give a more representative PSNR/SSIM than 128×128.
# Bicubic-only downsampling for a clean apples-to-apples comparison.
val_loader = get_dataloader(
    hr_dir='../data/DIV2K/DIV2K_valid_HR',
    batch_size=4,
    patch_size=256,
    fixed_scale=2,
    downsample_methods=['bicubic'],
    num_workers=2,
)

sr_psnr = sr_ssim = bic_psnr = bic_ssim = 0.0
n = 0

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        lr = batch['lr'].to(device)
        hr = batch['hr'].to(device)

        sr  = model(lr).clamp(0, 1)
        bic = F.interpolate(
            lr, size=hr.shape[2:], mode='bicubic', align_corners=False
        ).clamp(0, 1)

        sr_psnr  += fast_psnr(sr,  hr)
        sr_ssim  += fast_ssim(sr,  hr)
        bic_psnr += fast_psnr(bic, hr)
        bic_ssim += fast_ssim(bic, hr)
        n += 1

        if (i + 1) % 5 == 0 or i == len(val_loader) - 1:
            print(f'  [{i+1:3d}/{len(val_loader)}]  '
                  f'SR {sr_psnr/n:.2f} dB / {sr_ssim/n:.4f}   '
                  f'Bicubic {bic_psnr/n:.2f} dB / {bic_ssim/n:.4f}')

# ── Final results ────────────────────────────────────────────────────────────
print()
print(f'{"Method":<14} {"PSNR (dB)":>10} {"SSIM":>8}')
print('-' * 34)
print(f'{"Bicubic":<14} {bic_psnr/n:>10.2f} {bic_ssim/n:>8.4f}')
print(f'{"SR (ours)":<14} {sr_psnr/n:>10.2f} {sr_ssim/n:>8.4f}')
print(f'{"Δ gain":<14} {(sr_psnr - bic_psnr)/n:>+10.2f} {(sr_ssim - bic_ssim)/n:>+8.4f}')

In [ ]:
# ── Cell 7: Plot training history ────────────────────────────────────────────
# Displays the loss / PSNR / SSIM curves saved by train.py.
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DRIVE_DIR      = '/content/drive/MyDrive/COSC83_SR_scale2'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
history_png    = os.path.join(CHECKPOINT_DIR, 'training_history.png')

if os.path.exists(history_png):
    img = mpimg.imread(history_png)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'No training_history.png found at {history_png}')
    print('Run the Train cell to completion first.')